# Fabric Workspace Inventory (Scan API)

Extracts comprehensive object-level information for all Microsoft Fabric workspaces under the target capacities using the **Admin Workspace Scan API** (`getInfo` / `scanResult`).

Compared to the per-item API approach (`fabric_inventory.ipynb`), this notebook uses a batch scan that returns a global `datasourceInstances` array — providing complete connection details for **all** datasource types including ODBC, JDBC, OData, and gateway-bound sources.

**Notebook structure (modular cells):**

| Cell | Purpose |
|------|---------|
| 2 | Imports |
| 3 | Configuration — target capacity names |
| 4 | `admin_api_get()` / `admin_api_post()` — Admin API helpers with retry |
| 5 | `resolve_capacities()` — map capacity names → IDs |
| 6 | `fetch_workspace_ids()` — retrieve workspace IDs via Admin API |
| 7 | `scan_workspaces()` — run the async workspace scan (getInfo → scanStatus → scanResult) |
| 8 | `build_datasource_lookup()` / `resolve_datasources()` — map datasourceUsages to connection details |
| 9 | `format_permissions()` — format user permission strings |
| 10 | `get_refresh_schedule()` / `get_refresh_history()` — refresh metadata |
| 11 | `process_scan_workspaces()` — build inventory rows from scan results |
| 12 | `fetch_user_activity()` — Admin activity events with multi-day support |
| 13 | `get_fabric_inventory_scan()` — orchestrator |
| 14 | `export_to_onelake()` — export DataFrames to timestamped CSVs in OneLake |
| 15 | Execute, display results & export to OneLake |

**Prerequisites:** Fabric Administrator role + tenant settings (see README).

**Additional tenant setting required:** *Enhance admin APIs responses with detailed metadata* must be enabled for the scan API to return datasource details.

In [ ]:
import sempy.fabric as fabric
import pandas as pd
import json
import time
import os
from datetime import datetime, timedelta
from typing import Optional

In [ ]:
# ── Configuration ──

target_capacity_names = [
    "lakfabuswest2"
]

SCAN_POLL_INTERVAL_SECONDS = 3
SCAN_MAX_WAIT_SECONDS = 300
SCAN_BATCH_SIZE = 100  # max workspace IDs per scan request

In [ ]:
# ── Admin API helpers with pagination & retry ──

MAX_RETRIES = 3
DEFAULT_RETRY_SECONDS = 5


def _handle_rate_limit(response, url: str, retries: int) -> int:
    """Handle 429 rate-limit responses. Returns updated retry count."""
    retries += 1
    if retries > MAX_RETRIES:
        raise Exception(f"Rate-limited {MAX_RETRIES} times on {url}")
    wait = int(response.headers.get("Retry-After", DEFAULT_RETRY_SECONDS))
    print(f"    Rate limited – retrying in {wait}s (attempt {retries}/{MAX_RETRIES})…")
    time.sleep(wait)
    return retries


def admin_api_get(client, url: str, paginated: bool = False):
    """
    Call a Fabric Admin REST API GET endpoint.

    Args:
        client:    PowerBIRestClient instance.
        paginated: If True, follows continuationUri / @odata.nextLink.

    Returns:
        dict (single-page) or list[dict] (paginated).
    """
    results = []
    retries = 0

    while url:
        response = client.get(url)

        if response.status_code == 429:
            retries = _handle_rate_limit(response, url, retries)
            continue

        if response.status_code != 200:
            raise Exception(f"{response.status_code} {response.reason} for url: {url}")

        data = response.json()
        retries = 0

        if paginated:
            results.extend(data.get("value", []))
            url = data.get("continuationUri") or data.get("@odata.nextLink")
        else:
            return data

    return results


def admin_api_post(client, url: str, body: dict) -> dict:
    """
    Call a Fabric Admin REST API POST endpoint with retry on rate-limit.

    Returns:
        Parsed JSON response.
    """
    retries = 0
    while True:
        response = client.post(url, json=body)

        if response.status_code == 429:
            retries = _handle_rate_limit(response, url, retries)
            continue

        if response.status_code not in (200, 202):
            raise Exception(f"{response.status_code} {response.reason} for url: {url}")

        return response.json()

In [ ]:
# ── Resolve target capacities ──

def resolve_capacities(capacity_names: list[str]) -> tuple[dict, set]:
    """
    Map capacity display names to IDs.

    Returns:
        id_to_name: dict mapping lowercase capacity-id → display name.
        target_ids: set of lowercase capacity-ids matching *capacity_names*.
    """
    caps_df = fabric.list_capacities()
    lower_targets = {n.lower() for n in capacity_names}

    id_to_name = {
        str(row["Id"]).lower(): row["Display Name"]
        for _, row in caps_df.iterrows()
    }
    target_ids = {
        cid for cid, name in id_to_name.items()
        if name.lower() in lower_targets
    }

    if not target_ids:
        available = list(id_to_name.values())
        raise ValueError(
            f"No capacities matched {capacity_names}. Available: {available}"
        )

    print(f"  Matched {len(target_ids)} capacity ID(s) for {capacity_names}")
    return id_to_name, target_ids

In [ ]:
# ── Fetch workspace IDs via Admin API ──

def fetch_workspace_ids(client, target_ids: set[str]) -> list[dict]:
    """
    Fetch all tenant workspaces using the Admin API and filter to
    those belonging to *target_ids* capacities.

    Returns a lightweight list of {id, name, capacityId} dicts.
    The full metadata comes from the scan API.
    """
    print("  Fetching workspace list (Admin API)…")
    all_ws = admin_api_get(
        client,
        "/v1.0/myorg/admin/groups?$top=5000",
        paginated=True,
    )
    filtered = [
        {"id": ws["id"], "name": ws.get("name", "Unnamed"), "capacityId": ws.get("capacityId", "")}
        for ws in all_ws
        if str(ws.get("capacityId", "")).lower() in target_ids
    ]
    print(f"  {len(filtered)} workspaces matched target capacities.")
    if not filtered:
        raise ValueError(
            "No workspaces found for the target capacities. "
            "Verify capacity names and admin permissions."
        )
    return filtered

In [ ]:
# ── Workspace Scan API (getInfo → scanStatus → scanResult) ──

def scan_workspaces(
    client,
    workspace_ids: list[str],
    datasource_details: bool = True,
    dataset_expressions: bool = True,
    get_artifact_users: bool = True,
) -> dict:
    """
    Run the Admin Workspace Scan API to get detailed metadata
    including datasource instances (ODBC, JDBC, etc.).

    Handles batching (max 100 workspace IDs per request) and
    polls scanStatus until completion.

    Returns:
        Combined scan result with 'workspaces' and 'datasourceInstances' keys.
    """
    combined = {"workspaces": [], "datasourceInstances": []}

    # Build query parameters
    params = []
    if datasource_details:
        params.append("datasourceDetails=True")
    if dataset_expressions:
        params.append("datasetExpressions=True")
    if get_artifact_users:
        params.append("getArtifactUsers=True")
    qs = "&".join(params)

    for batch_start in range(0, len(workspace_ids), SCAN_BATCH_SIZE):
        batch = workspace_ids[batch_start : batch_start + SCAN_BATCH_SIZE]
        batch_num = batch_start // SCAN_BATCH_SIZE + 1
        total_batches = (len(workspace_ids) + SCAN_BATCH_SIZE - 1) // SCAN_BATCH_SIZE
        print(f"  Scan batch {batch_num}/{total_batches} ({len(batch)} workspaces)…")

        # 1. Initiate the scan
        scan_resp = admin_api_post(
            client,
            f"/v1.0/myorg/admin/workspaces/getInfo?{qs}",
            {"workspaces": batch},
        )
        scan_id = scan_resp.get("id")
        if not scan_id:
            raise Exception(f"Scan initiation failed — no scan ID returned: {scan_resp}")

        # 2. Poll for completion
        elapsed = 0
        while elapsed < SCAN_MAX_WAIT_SECONDS:
            status_resp = admin_api_get(
                client, f"/v1.0/myorg/admin/workspaces/scanStatus/{scan_id}"
            )
            status = status_resp.get("status", "Unknown")
            if status == "Succeeded":
                break
            if status in ("Failed", "NotFound"):
                raise Exception(f"Scan {scan_id} failed with status: {status} — {status_resp}")
            time.sleep(SCAN_POLL_INTERVAL_SECONDS)
            elapsed += SCAN_POLL_INTERVAL_SECONDS
        else:
            raise Exception(f"Scan {scan_id} timed out after {SCAN_MAX_WAIT_SECONDS}s")

        # 3. Retrieve results
        result = admin_api_get(
            client, f"/v1.0/myorg/admin/workspaces/scanResult/{scan_id}"
        )
        combined["workspaces"].extend(result.get("workspaces", []))
        combined["datasourceInstances"].extend(result.get("datasourceInstances", []))

    print(f"  Scan complete: {len(combined['workspaces'])} workspaces, "
          f"{len(combined['datasourceInstances'])} datasource instances.")
    return combined

In [ ]:
# ── Datasource lookup from scan results ──

def build_datasource_lookup(scan_result: dict) -> dict[str, dict]:
    """
    Build a lookup from datasourceId → datasource instance dict.

    The scan result contains a top-level 'datasourceInstances' array
    with entries like:
        {
            "datasourceType": "Sql",
            "connectionDetails": {"server": "...", "database": "..."},
            "datasourceId": "guid",
            "gatewayId": "guid"
        }
    """
    return {
        ds["datasourceId"]: ds
        for ds in scan_result.get("datasourceInstances", [])
        if "datasourceId" in ds
    }


def _format_connection(source: dict) -> str:
    """Build a human-readable connection string from a datasource instance."""
    details = source.get("connectionDetails") or {}
    if isinstance(details, str):
        try:
            details = json.loads(details)
        except (json.JSONDecodeError, TypeError):
            details = {}
    ds_type = source.get("datasourceType", "Unknown")
    server = details.get("server", "")
    database = details.get("database", "")
    if server:
        return f"{ds_type}: {server}/{database}"
    path = details.get("path", "")
    url = details.get("url", "")
    conn_string = details.get("connectionString", "")
    return f"{ds_type}: {path or url or conn_string or 'N/A'}"


def resolve_datasources(
    item: dict,
    ds_lookup: dict[str, dict],
) -> tuple[str, str]:
    """
    Resolve connection details and datasource types for an item
    using its datasourceUsages and the global datasource lookup.

    Returns:
        (connection_details, datasource_type) tuple.
    """
    usages = item.get("datasourceUsages", [])
    if not usages:
        return "No datasources", "N/A"

    sources = [
        ds_lookup[u["datasourceInstanceId"]]
        for u in usages
        if u.get("datasourceInstanceId") in ds_lookup
    ]

    if not sources:
        # datasourceUsages exist but IDs not in the lookup (misconfigured)
        misconfig = item.get("misconfiguredDatasourceUsages", [])
        if misconfig:
            return "Misconfigured datasource(s)", "Unknown"
        return "No datasources", "N/A"

    conn_strings = list(dict.fromkeys(_format_connection(s) for s in sources))
    ds_types = list(dict.fromkeys(s.get("datasourceType", "Unknown") for s in sources))
    return "; ".join(conn_strings), "; ".join(ds_types)

In [ ]:
# ── Extract user permissions ──

def format_permissions(users: list[dict]) -> str:
    """Format the user list from the scan result."""
    if not users:
        return "No users found"

    entries = []
    for u in users:
        identifier = (
            u.get("emailAddress")
            or u.get("displayName")
            or u.get("identifier", "Unknown")
        )
        role = u.get("groupUserAccessRight", u.get("principalType", "Unknown"))
        p_type = u.get("principalType", "")
        entries.append(f"{identifier} ({role}, {p_type})")
    return " | ".join(entries)

In [ ]:
# ── Extract refresh schedule & history ──

def get_refresh_schedule(client, dataset_id: str) -> str:
    """Return a formatted refresh-schedule string for a semantic model."""
    try:
        sched = admin_api_get(client, f"/v1.0/myorg/admin/datasets/{dataset_id}/refreshSchedule")
        if not sched:
            return "N/A"
        if not sched.get("enabled"):
            return "Disabled"
        freq = sched.get("frequency", "Unknown")
        times = ", ".join(sched.get("times", []))
        tz = sched.get("localTimeZoneId", "")
        return f"Every {freq} at [{times}] ({tz})"
    except Exception as e:
        if "401" in str(e) or "404" in str(e):
            return "N/A (not refreshable)"
        return f"Error: {str(e)[:80]}"


def get_refresh_history(client, dataset_id: str, top: int = 3, errors: list | None = None) -> str:
    """Return the last *top* refresh runs as a formatted string."""
    try:
        resp = admin_api_get(client, f"/v1.0/myorg/admin/datasets/{dataset_id}/refreshes?$top={top}")
        runs = resp.get("value", []) if isinstance(resp, dict) else resp
        if not runs:
            return "No history"
        entries = []
        for h in runs[:top]:
            status = h.get("status", "Unknown")
            start = h.get("startTime", "")[:19]
            end = h.get("endTime", "")[:19]
            entries.append(f"{status} ({start} → {end})")
        return " | ".join(entries)
    except Exception as e:
        if "401" in str(e) or "404" in str(e):
            return "N/A (not refreshable)"
        if errors is not None:
            errors.append({"Item": dataset_id, "Type": "RefreshHistory", "Error": str(e)[:200]})
        return "N/A"

In [ ]:
# ── Build inventory rows from scan results ──

def process_scan_workspaces(
    client,
    scan_result: dict,
    id_to_name: dict,
    errors: list,
) -> list[dict]:
    """
    Process scan results into flat inventory rows.

    Uses the global datasourceInstances lookup to resolve connection
    details for all item types (semantic models, dataflows, reports).
    """
    ds_lookup = build_datasource_lookup(scan_result)
    rows = []

    for ws in scan_result.get("workspaces", []):
        ws_id = ws.get("id", "")
        ws_name = ws.get("name", "Unnamed")
        cap_name = id_to_name.get(str(ws.get("capacityId", "")).lower(), "Unknown")
        perm = format_permissions(ws.get("users", []))
        print(f"    {ws_name}…")

        base = {
            "CapacityName": cap_name,
            "WorkspaceName": ws_name,
            "WorkspaceId": ws_id,
            "UserPermissions": perm,
        }

        # ── Semantic models ──
        for ds in ws.get("datasets", []):
            ds_id = ds.get("id", "N/A")
            ds_name = ds.get("name", "Unnamed")
            conn, ds_type = resolve_datasources(ds, ds_lookup)
            sched = get_refresh_schedule(client, ds_id)
            hist = get_refresh_history(client, ds_id, errors=errors)

            rows.append({
                **base,
                "ItemName": ds_name,
                "ItemType": "SemanticModel",
                "ItemId": ds_id,
                "DataSourceType": ds_type,
                "ConnectionDetails": conn,
                "RefreshSchedule": sched,
                "RefreshHistory": hist,
                "StorageMode": ds.get("targetStorageMode", "N/A"),
                "ConfiguredBy": ds.get("configuredBy", "N/A"),
            })

        # ── Dataflows ──
        for df in ws.get("dataflows", []):
            df_id = df.get("objectId", df.get("id", "N/A"))
            df_name = df.get("name", "Unnamed")
            conn, ds_type = resolve_datasources(df, ds_lookup)

            rows.append({
                **base,
                "ItemName": df_name,
                "ItemType": "Dataflow",
                "ItemId": df_id,
                "DataSourceType": ds_type,
                "ConnectionDetails": conn,
                "RefreshSchedule": "N/A",
                "RefreshHistory": "N/A",
                "StorageMode": "N/A",
                "ConfiguredBy": df.get("configuredBy", "N/A"),
            })

        # ── Reports ──
        for rpt in ws.get("reports", []):
            rpt_id = rpt.get("id", "N/A")
            rpt_name = rpt.get("name", "Unnamed")
            dataset_id = rpt.get("datasetId", "N/A")

            rows.append({
                **base,
                "ItemName": rpt_name,
                "ItemType": "Report",
                "ItemId": rpt_id,
                "DataSourceType": "N/A (Report)",
                "ConnectionDetails": f"Bound to dataset: {dataset_id}",
                "RefreshSchedule": "N/A (Report)",
                "RefreshHistory": "N/A (Report)",
                "StorageMode": "N/A",
                "ConfiguredBy": rpt.get("createdBy", "N/A"),
            })

    return rows

In [ ]:
# ── Fetch user activity events ──

def fetch_user_activity(
    client,
    workspace_ids: set[str],
    lookback_days: int = 1,
    errors: list | None = None,
) -> list[dict]:
    """
    Fetch user activity events from the Admin API for a 1-day window
    and filter to the given workspace IDs.

    The Admin activity API only supports 1-day windows, so we iterate
    over the last *lookback_days* days.
    """
    activity_rows = []
    now = datetime.utcnow().replace(hour=0, minute=0, second=0)

    for day_offset in range(lookback_days):
        query_date = now - timedelta(days=day_offset + 1)
        start_str = query_date.strftime("%Y-%m-%dT00:00:00")
        end_str = query_date.strftime("%Y-%m-%dT23:59:59")

        try:
            events = admin_api_get(
                client,
                f"/v1.0/myorg/admin/activityevents?startDateTime=%27{start_str}%27&endDateTime=%27{end_str}%27",
                paginated=True,
            )
            for act in events:
                if act.get("WorkspaceId") in workspace_ids:
                    activity_rows.append({
                        "WorkspaceName": act.get("WorkspaceName", ""),
                        "Operation": act.get("Operation", ""),
                        "User": act.get("UserId", ""),
                        "ItemName": act.get("ArtifactName", act.get("DatasetName", "")),
                        "Timestamp": act.get("CreationTime", ""),
                    })
        except Exception as e:
            msg = f"Activity day {query_date.date()}: {e}"
            print(f"    ⚠ {msg}")
            if errors is not None:
                errors.append({"Item": "ActivityEvents", "Type": "UserActivity", "Error": str(msg)[:200]})

    print(f"  {len(activity_rows)} activity events captured ({lookback_days}-day window).")
    return activity_rows

In [ ]:
# ── Orchestrator: run full inventory extraction via Scan API ──

def get_fabric_inventory_scan(
    capacity_names: list[str],
    workspace_names: list[str] | None = None,
    activity_lookback_days: int = 1,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    End-to-end inventory extraction using the Admin Workspace Scan API.

    This uses the async scan flow (getInfo → scanStatus → scanResult)
    which returns a global datasourceInstances array covering all
    datasource types including ODBC, JDBC, and gateway-bound sources.

    Args:
        capacity_names:          List of capacity display names to scan.
        workspace_names:         Optional list to limit the scan.
        activity_lookback_days:  Days of user-activity to fetch (max 30).

    Returns:
        (inventory_df, activity_df, errors_df)
    """
    client = fabric.PowerBIRestClient()
    errors: list[dict] = []

    # 1. Capacities
    print("Step 1: Resolving capacities…")
    id_to_name, target_ids = resolve_capacities(capacity_names)

    # 2. Workspace IDs
    print("Step 2: Fetching workspace list…")
    workspaces = fetch_workspace_ids(client, target_ids)

    # 2b. Filter to specific workspaces if requested
    if workspace_names:
        lower_ws = {n.lower() for n in workspace_names}
        workspaces = [ws for ws in workspaces if ws["name"].lower() in lower_ws]
        if not workspaces:
            raise ValueError(
                f"No workspaces matched {workspace_names} in the target capacities."
            )
        print(f"  Filtered to {len(workspaces)} workspace(s): {workspace_names}")

    # 3. Scan
    print("Step 3: Running workspace scan (getInfo → scanResult)…")
    ws_id_list = [ws["id"] for ws in workspaces]
    scan_result = scan_workspaces(client, ws_id_list)

    # 4. Inventory
    print("Step 4: Building item inventory from scan results…")
    inventory_rows = process_scan_workspaces(client, scan_result, id_to_name, errors)

    # 5. Activity
    print("Step 5: Fetching user activity…")
    ws_ids = set(ws_id_list)
    activity_rows = fetch_user_activity(client, ws_ids, activity_lookback_days, errors)

    return (
        pd.DataFrame(inventory_rows),
        pd.DataFrame(activity_rows),
        pd.DataFrame(errors),
    )

In [ ]:
# ── Export DataFrame to OneLake as CSV ──

def export_to_onelake(
    df: pd.DataFrame,
    prefix: str = "fabric_items",
    lakehouse_path: str = "/lakehouse/default/Files/csv_exports",
) -> str:
    """
    Export a DataFrame to a timestamped CSV file in OneLake.

    Args:
        df:             The DataFrame to export.
        prefix:         Filename prefix.
        lakehouse_path: OneLake path (default: attached lakehouse Files folder).

    Returns:
        The full path of the written CSV file.
    """
    os.makedirs(lakehouse_path, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prefix}_{timestamp}.csv"
    full_path = f"{lakehouse_path}/{filename}"
    df.to_csv(full_path, index=False)
    print(f"  Exported {len(df)} rows → {full_path}")
    return full_path

In [ ]:
# ── Execute extraction ──

inventory_df, activity_df, errors_df = get_fabric_inventory_scan(
    capacity_names=target_capacity_names,
    #workspace_names=["MyTestWorkspace"],  # ← uncomment to scan a single workspace
    workspace_names=None,  # ← set to None for all workspaces
    activity_lookback_days=1,
)

print(f"\n{'=' * 60}")
print("INVENTORY SUMMARY")
print(f"{'=' * 60}")
print(f"Total items: {len(inventory_df)}")

if not inventory_df.empty:
    print(f"\nBy type:\n{inventory_df['ItemType'].value_counts().to_string()}")
    print(f"\nBy capacity:\n{inventory_df['CapacityName'].value_counts().to_string()}")
    print(f"\nBy datasource type:\n{inventory_df['DataSourceType'].value_counts().to_string()}")

print(f"\nActivity events: {len(activity_df)}")

if not errors_df.empty:
    print(f"\n⚠ {len(errors_df)} errors encountered:")
    display(errors_df)

print(f"\n{'=' * 60}")
print("FULL INVENTORY:")
display(inventory_df)

if not activity_df.empty:
    print("\nUSER ACTIVITY:")
    display(activity_df)

# ── Export to OneLake ──
print(f"\n{'=' * 60}")
print("EXPORTING TO ONELAKE…")
if not inventory_df.empty:
    export_to_onelake(inventory_df, prefix="fabric_items_scan")
else:
    print("  Skipped inventory — DataFrame is empty.")

if not activity_df.empty:
    export_to_onelake(activity_df, prefix="fabric_activity_scan")
else:
    print("  Skipped activity — DataFrame is empty.")

if not errors_df.empty:
    export_to_onelake(errors_df, prefix="fabric_errors_scan")
else:
    print("  No errors to export.")